# Trivago Recommender System – Data Preprocessing

This notebook prepares the Trivago session and candidate-level data for modelling following the exploratory data analysis. It covers the construction of session-based behavioural features, train/validation splitting, feature selection and engineering, missing-value treatment, and preparation of model-specific inputs.

For each ranking event, only information available before or at that event is used to construct predictors, preventing subsequent session interactions from introducing target leakage.

In [ ]:
# Core libraries
import pandas as pd
import numpy as np

# Reproducibility
RANDOM_STATE = 42

In [ ]:
# Load data from the EDA checkpoints

from pathlib import Path

DATA_PATH = Path(".")

clickouts = pd.read_parquet(
    f"{DATA_PATH}/clickouts_sample.parquet"
)

metadata = pd.read_parquet(
    f"{DATA_PATH}/metadata.parquet"
)

candidates = pd.read_parquet(
    f"{DATA_PATH}/candidates.parquet"
)

sampled_train_sessions = pd.read_parquet(
    f"{DATA_PATH}/train_session_sample_10pct.parquet"
)

session_events = pd.read_parquet(
    f"{DATA_PATH}/session_events_sample.parquet"
)

print("EDA checkpoints loaded successfully.")

EDA checkpoints loaded successfully.


In [ ]:
# Verify candidate-level modelling dataset

print("Shape:", candidates.shape)
print("\nColumns:")
print(candidates.columns.tolist())

print("\nRanking events:", candidates["ranking_event_id"].nunique())
print("Clicked candidates:", candidates["clicked"].sum())
print("Overall click rate:", candidates["clicked"].mean())

Shape: (3616016, 24)

Columns:
['user_id', 'session_id', 'timestamp', 'step', 'action_type', 'reference', 'platform', 'city', 'device', 'current_filters', 'impressions', 'prices', 'ranking_event_id', 'candidate_id', 'candidate_price', 'candidate_position', 'clicked', 'item_id', 'properties', 'num_properties', 'selected_filters', 'filter_match', 'impression_median_price', 'relative_price']

Ranking events: 157809
Clicked candidates: 157726
Overall click rate: 0.04361872292600475


In [ ]:
# Check ranking events without exactly one clicked candidate

clicks_per_event = (
    candidates
    .groupby("ranking_event_id")["clicked"]
    .sum()
)

print("Ranking events:", len(clicks_per_event))
print("Events with exactly 1 click:", (clicks_per_event == 1).sum())
print("Events with 0 clicks:", (clicks_per_event == 0).sum())
print("Events with >1 click:", (clicks_per_event > 1).sum())

Ranking events: 157809
Events with exactly 1 click: 157726
Events with 0 clicks: 83
Events with >1 click: 0


In [ ]:
# Check whether the clicked item is absent from the candidate list

zero_click_event_ids = clicks_per_event[
    clicks_per_event == 0
].index

zero_click_clickouts = clickouts[
    clickouts["ranking_event_id"].isin(zero_click_event_ids)
].copy()

zero_click_clickouts["clicked_item_in_impressions"] = (
    zero_click_clickouts.apply(
        lambda row: str(row["reference"]) in str(row["impressions"]).split("|"),
        axis=1
    )
)

print(
    zero_click_clickouts["clicked_item_in_impressions"]
    .value_counts(dropna=False)
)

clicked_item_in_impressions
False    83
Name: count, dtype: int64


In [ ]:
# Remove ranking events where the clicked item is absent from the candidate set

valid_ranking_event_ids = clicks_per_event[
    clicks_per_event == 1
].index

candidates = candidates[
    candidates["ranking_event_id"].isin(valid_ranking_event_ids)
].copy()

print("Candidate rows:", len(candidates))
print("Ranking events:", candidates["ranking_event_id"].nunique())
print("Clicked candidates:", candidates["clicked"].sum())

Candidate rows: 3614130
Ranking events: 157726
Clicked candidates: 157726


In [ ]:
# Checkpoint: save candidates with valid ranking targets

candidates.to_parquet(
    f"{DATA_PATH}/candidates_valid.parquet",
    index=False
)

print("Valid candidate checkpoint saved.")
print("Candidate rows:", len(candidates))
print("Ranking events:", candidates["ranking_event_id"].nunique())
print("Clicked candidates:", candidates["clicked"].sum())

Valid candidate checkpoint saved.
Candidate rows: 3614130
Ranking events: 157726
Clicked candidates: 157726


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# Split complete sessions into training, validation and test sets

session_ids = candidates["session_id"].unique()

train_sessions, temp_sessions = train_test_split(
    session_ids,
    test_size=0.30,
    random_state=RANDOM_STATE
)

val_sessions, test_sessions = train_test_split(
    temp_sessions,
    test_size=0.50,
    random_state=RANDOM_STATE
)

train_data = candidates[
    candidates["session_id"].isin(train_sessions)
].copy()

val_data = candidates[
    candidates["session_id"].isin(val_sessions)
].copy()

test_data = candidates[
    candidates["session_id"].isin(test_sessions)
].copy()

In [ ]:
# Validate train/validation/test split

train_session_ids = set(train_data["session_id"])
val_session_ids = set(val_data["session_id"])
test_session_ids = set(test_data["session_id"])

train_event_ids = set(train_data["ranking_event_id"])
val_event_ids = set(val_data["ranking_event_id"])
test_event_ids = set(test_data["ranking_event_id"])

print("Session overlap:")
print("Train / validation:", len(train_session_ids & val_session_ids))
print("Train / test:", len(train_session_ids & test_session_ids))
print("Validation / test:", len(val_session_ids & test_session_ids))

print("\nRanking-event overlap:")
print("Train / validation:", len(train_event_ids & val_event_ids))
print("Train / test:", len(train_event_ids & test_event_ids))
print("Validation / test:", len(val_event_ids & test_event_ids))

print("\nSessions:")
print("Training:", train_data["session_id"].nunique())
print("Validation:", val_data["session_id"].nunique())
print("Test:", test_data["session_id"].nunique())

print("\nRanking events:")
print("Training:", train_data["ranking_event_id"].nunique())
print("Validation:", val_data["ranking_event_id"].nunique())
print("Test:", test_data["ranking_event_id"].nunique())

print("\nClick rates:")
print("Training:", train_data["clicked"].mean())
print("Validation:", val_data["clicked"].mean())
print("Test:", test_data["clicked"].mean())

print("\nClicked candidates:")
print("Training:", train_data["clicked"].sum())
print("Validation:", val_data["clicked"].sum())
print("Test:", test_data["clicked"].sum())

Session overlap:
Train / validation: 0
Train / test: 0
Validation / test: 0

Ranking-event overlap:
Train / validation: 0
Train / test: 0
Validation / test: 0

Sessions:
Training: 57849
Validation: 12396
Test: 12397

Ranking events:
Training: 110716
Validation: 23737
Test: 23273

Click rates:
Training: 0.043596700202791834
Validation: 0.043692823089808014
Test: 0.043803052831680185

Clicked candidates:
Training: 110716
Validation: 23737
Test: 23273


In [ ]:
# Checkpoint: save training candidate data

train_data.to_parquet(
    f"{DATA_PATH}/train_candidates.parquet",
    index=False
)

print("Training checkpoint saved.")
print("Rows:", len(train_data))
print("Sessions:", train_data["session_id"].nunique())
print("Ranking events:", train_data["ranking_event_id"].nunique())

Training checkpoint saved.
Rows: 2539550
Sessions: 57849
Ranking events: 110716


In [ ]:
# Checkpoint: save validation candidate data

val_data.to_parquet(
    f"{DATA_PATH}/val_candidates.parquet",
    index=False
)

print("Validation checkpoint saved.")
print("Rows:", len(val_data))
print("Sessions:", val_data["session_id"].nunique())
print("Ranking events:", val_data["ranking_event_id"].nunique())

Validation checkpoint saved.
Rows: 543270
Sessions: 12396
Ranking events: 23737


In [ ]:
# Checkpoint: save test candidate data

test_data.to_parquet(
    f"{DATA_PATH}/test_candidates.parquet",
    index=False
)

print("Test checkpoint saved.")
print("Rows:", len(test_data))
print("Sessions:", test_data["session_id"].nunique())
print("Ranking events:", test_data["ranking_event_id"].nunique())

Test checkpoint saved.
Rows: 531310
Sessions: 12397
Ranking events: 23273


## Feature selection

Features identified during the EDA are organised according to their role in the modelling process.

- **Target:** `clicked`
- **Grouping variable:** `ranking_event_id`
- **Session grouping variable:** `session_id`
- **Candidate features:** `candidate_price`, `relative_price`, `properties`, `filter_match`
- **Search/context features:** `city`, `platform`, `device`
- **Session behavioural features:** features derived from interactions in the behavioural sequence preceding each ranking event, including searches, filter selections and interactions with accommodation items
- **Evaluation/benchmark feature:** `candidate_position`

The session behavioural features will be constructed using only interactions occurring before the ranking event. This allows the models to incorporate the user's evolving within-session behaviour while avoiding information from subsequent interactions that would introduce target leakage.

`candidate_position` is retained for evaluation but excluded from model training because it represents the position assigned by Trivago's existing ranking system. Including it as a predictor could therefore cause the models to reproduce the existing ranking and introduce exposure bias.

In [ ]:
# Examine missing values in selected modelling features

selected_features = [
    "candidate_price",
    "relative_price",
    "properties",
    "filter_match",
    "city",
    "platform",
    "device"
]

missing_values = train_data[selected_features].isna().sum()
missing_percentage = (
    train_data[selected_features].isna().mean() * 100
)

missing_summary = pd.DataFrame({
    "missing_values": missing_values,
    "missing_percentage": missing_percentage
})

missing_summary

,missing_values,missing_percentage
candidate_price,0,0.000000
relative_price,0,0.000000
properties,929,0.036581
filter_match,0,0.000000
city,0,0.000000
platform,0,0.000000
device,0,0.000000


### Missing values

Missing-value analysis showed that most selected features contained no missing values. Candidate `properties` were missing for only 0.037% of training observations.

In [ ]:
# Represent meaningful missing categorical values explicitly

train_data["properties"] = train_data["properties"].fillna("Unknown")
val_data["properties"] = val_data["properties"].fillna("Unknown")
test_data["properties"] = test_data["properties"].fillna("Unknown")

In [ ]:
# Verify missing values after treatment

print("TRAIN")
print(train_data[selected_features].isna().sum())

print("\nVALIDATION")
print(val_data[selected_features].isna().sum())

print("\nTEST")
print(test_data[selected_features].isna().sum())

TRAIN
candidate_price    0
relative_price     0
properties         0
filter_match       0
city               0
platform           0
device             0
dtype: int64

VALIDATION
candidate_price    0
relative_price     0
properties         0
filter_match       0
city               0
platform           0
device             0
dtype: int64

TEST
candidate_price    0
relative_price     0
properties         0
filter_match       0
city               0
platform           0
device             0
dtype: int64


In [ ]:
# Inspect multi-value feature

print("\nPROPERTIES")
print(train_data["properties"].head(10).to_string(index=False))


PROPERTIES
Horse Riding|Cosmetic Mirror|3 Star|Sauna|WiFi ...
Openable Windows|Sitting Area (Rooms)|WiFi (Pub...
Sauna|Electric Kettle|Microwave|Good Rating|Pet...
Desk|WiFi (Public Areas)|Conference Rooms|Resta...
Radio|Laundry Service|3 Star|From 2 Stars|Deck ...
Sauna|Safe (Hotel)|Telephone|Hotel|Cable TV|Sun...
                                 House / Apartment
From 2 Stars|Tennis Court (Indoor)|Radio|4 Star...
Electric Kettle|Country Hotel|Satisfactory Rati...
Cot|Hairdryer|Ski Resort|Accessible Parking|Fro...


In [ ]:
# Examine vocabulary size of multi-value features

property_values = set()
for value in train_data.loc[
    train_data["properties"] != "Unknown",
    "properties"
]:
    property_values.update(value.split("|"))

print("Unique property values:", len(property_values))

Unique property values: 157


In [ ]:
# Save train, validation and test base preprocessing checkpoints

train_data.to_parquet(
    f"{DATA_PATH}/train_preprocessed_base.parquet",
    index=False
)

val_data.to_parquet(
    f"{DATA_PATH}/val_preprocessed_base.parquet",
    index=False
)

test_data.to_parquet(
    f"{DATA_PATH}/test_preprocessed_base.parquet",
    index=False
)

print("Base preprocessing checkpoints saved.")

Base preprocessing checkpoints saved.


## Model inputs

The target variable is `clicked`, indicating whether a candidate accommodation was selected within its ranking event. `ranking_event_id` is retained as a grouping variable for ranking models and evaluation but is not used as a predictive feature. `session_id` identifies the session containing each ranking event and is used to preserve session-level separation between the training, validation and test datasets.

The predictors include candidate, search/context and preceding session-behaviour information. Session-behaviour features are constructed using only interactions occurring before the relevant ranking event. The resulting predictors are separated into numerical, categorical and multi-value features. Model-specific encoding and scaling will be applied subsequently because the selected model architectures have different preprocessing requirements.

In [ ]:
# Define modelling variables

target = "clicked"
grouping_feature = "ranking_event_id"
session_grouping_feature = "session_id"

numerical_features = [
    "candidate_price",
    "relative_price",
    "filter_match"
]

categorical_features = [
    "city",
    "platform",
    "device"
]

multi_value_features = [
    "properties"
]

evaluation_features = [
    "candidate_position"
]

model_features = (
    numerical_features
    + categorical_features
    + multi_value_features
)

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)
print("Multi-value features:", multi_value_features)
print("Target:", target)
print("Grouping feature:", grouping_feature)
print("Session grouping feature:", session_grouping_feature)
print("Evaluation-only:", evaluation_features)

Numerical features: ['candidate_price', 'relative_price', 'filter_match']
Categorical features: ['city', 'platform', 'device']
Multi-value features: ['properties']
Target: clicked
Grouping feature: ranking_event_id
Session grouping feature: session_id
Evaluation-only: ['candidate_position']


In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline

In [ ]:
# Check the effect of different rare-city thresholds

city_counts = train_data["city"].value_counts()

for threshold in [2, 4, 10, 20, 50, 100]:
    retained = (city_counts >= threshold).sum()
    rare_rows = city_counts[city_counts < threshold].sum()

    print(
        f"Minimum {threshold:>3} occurrences: "
        f"{retained:>5} cities retained | "
        f"{rare_rows / len(train_data):.2%} of rows grouped as Other"
    )

Minimum   2 occurrences: 10209 cities retained | 0.00% of rows grouped as Other
Minimum   4 occurrences: 10052 cities retained | 0.02% of rows grouped as Other
Minimum  10 occurrences:  9527 cities retained | 0.15% of rows grouped as Other
Minimum  20 occurrences:  8493 cities retained | 0.73% of rows grouped as Other
Minimum  50 occurrences:  5104 cities retained | 4.51% of rows grouped as Other
Minimum 100 occurrences:  3041 cities retained | 9.78% of rows grouped as Other


Based on the above analysis, a minimum-frequency threshold of 50 was selected. This retains 5,104 city categories while grouping only 4.51% of training observations into Other. This reduces the dimensionality of subsequent categorical encoding while retaining cities with sufficient representation in the training data.

In [ ]:
# Group infrequent cities for Logistic Regression

CITY_MIN_FREQUENCY = 50

city_counts = train_data["city"].value_counts()

frequent_cities = city_counts[
    city_counts >= CITY_MIN_FREQUENCY
].index

train_data["city_lr"] = train_data["city"].where(
    train_data["city"].isin(frequent_cities),
    "Other"
)

val_data["city_lr"] = val_data["city"].where(
    val_data["city"].isin(frequent_cities),
    "Other"
)

test_data["city_lr"] = test_data["city"].where(
    test_data["city"].isin(frequent_cities),
    "Other"
)

print("Training city categories:", train_data["city_lr"].nunique())
print("Validation city categories:", val_data["city_lr"].nunique())
print("Test city categories:", test_data["city_lr"].nunique())

print(
    "Training rows grouped as Other:",
    (train_data["city_lr"] == "Other").mean()
)
print(
    "Validation rows grouped as Other:",
    (val_data["city_lr"] == "Other").mean()
)
print(
    "Test rows grouped as Other:",
    (test_data["city_lr"] == "Other").mean()
)

Training city categories: 5105
Validation city categories: 2537
Test city categories: 2472
Training rows grouped as Other: 0.04509972239176232
Validation rows grouped as Other: 0.10252360704622011
Test rows grouped as Other: 0.10458677608176017


## Session behavioural feature engineering

The following features are derived from the sequence of user interactions preceding each ranking event. Only interactions occurring before the ranking event are used, preventing subsequent behaviour from introducing target leakage.

In [ ]:
# Create counts of preceding interaction types for each behavioural sequence

behaviour_events = session_events.loc[
    ~session_events["is_clickout"]
].copy()

behaviour_counts = (
    behaviour_events
    .groupby(
        ["session_id", "behaviour_sequence", "action_type"]
    )
    .size()
    .unstack(fill_value=0)
)

behaviour_counts.head()

action_type                       change of sort order  filter selection  \
session_id    behaviour_sequence                                           
0002ccee5a980 0                                      0                 0   
0002d9458ab1c 1                                      0                 0   
00056fee0c63f 1                                      0                 0   
0005fe7961566 1                                      0                 0   
              2                                      0                 0   

action_type                       interaction item deals  \
session_id    behaviour_sequence                           
0002ccee5a980 0                                        0   
0002d9458ab1c 1                                        0   
00056fee0c63f 1                                        0   
0005fe7961566 1                                        0   
              2                                        0   

action_type                       interaction item image  \
session_id    behaviour_sequence                           
0002ccee5a980 0                                        2   
0002d9458ab1c 1                                        0   
00056fee0c63f 1                                        0   
0005fe7961566 1                                       25   
              2                                       20   

action_type                       interaction item info  \
session_id    behaviour_sequence                          
0002ccee5a980 0                                       0   
0002d9458ab1c 1                                       0   
00056fee0c63f 1                                       0   
0005fe7961566 1                                       0   
              2                                       0   

action_type                       interaction item rating  \
session_id    behaviour_sequence                            
0002ccee5a980 0                                         0   
0002d9458ab1c 1                                         0   
00056fee0c63f 1                                         1   
0005fe7961566 1                                         0   
              2                                         0   

action_type                       search for destination  search for item  \
session_id    behaviour_sequence                                            
0002ccee5a980 0                                        0                0   
0002d9458ab1c 1                                        0                1   
00056fee0c63f 1                                        0                0   
0005fe7961566 1                                        0                0   
              2                                        0                0   

action_type                       search for poi  
session_id    behaviour_sequence                  
0002ccee5a980 0                                0  
0002d9458ab1c 1                                0  
00056fee0c63f 1                                0  
0005fe7961566 1                                0  
              2                                0

In [ ]:
# Rename behavioural count columns for modelling

behaviour_counts = behaviour_counts.rename(columns={
    "change of sort order": "count_sort_change",
    "filter selection": "count_filter_selection",
    "interaction item deals": "count_item_deals",
    "interaction item image": "count_item_image",
    "interaction item info": "count_item_info",
    "interaction item rating": "count_item_rating",
    "search for destination": "count_search_destination",
    "search for item": "count_search_item",
    "search for poi": "count_search_poi"
}).reset_index()

print(behaviour_counts.columns.tolist())

['session_id', 'behaviour_sequence', 'count_sort_change', 'count_filter_selection', 'count_item_deals', 'count_item_image', 'count_item_info', 'count_item_rating', 'count_search_destination', 'count_search_item', 'count_search_poi']


In [ ]:
# Link each ranking event to its preceding behavioural sequence

ranking_behaviour = (
    session_events.loc[
        session_events["is_clickout"],
        ["session_id", "behaviour_sequence", "ranking_event_id"]
    ]
    .dropna(subset=["ranking_event_id"])
    .merge(
        behaviour_counts,
        on=["session_id", "behaviour_sequence"],
        how="left"
    )
)

print("Ranking events:", ranking_behaviour["ranking_event_id"].nunique())
print("Rows:", len(ranking_behaviour))
print("Duplicate ranking events:", ranking_behaviour["ranking_event_id"].duplicated().sum())

ranking_behaviour.head()

Ranking events: 157809
Rows: 157809
Duplicate ranking events: 0


,session_id,behaviour_sequence,ranking_event_id,count_sort_change,count_filter_selection,count_item_deals,count_item_image,count_item_info,count_item_rating,count_search_destination,count_search_item,count_search_poi
0,0002ccee5a980,0,0002ccee5a980_3,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
1,0002d9458ab1c,0,0002d9458ab1c_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0002d9458ab1c,0,0002d9458ab1c_2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00030e90e12b2,0,00030e90e12b2_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00030e90e12b2,0,00030e90e12b2_2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Replace missing behavioural counts with zero

behavioural_features = [
    "count_sort_change",
    "count_filter_selection",
    "count_item_deals",
    "count_item_image",
    "count_item_info",
    "count_item_rating",
    "count_search_destination",
    "count_search_item",
    "count_search_poi"
]

print(
    "Ranking events with no preceding behaviour:",
    ranking_behaviour[behavioural_features].isna().all(axis=1).sum()
)

ranking_behaviour[behavioural_features] = (
    ranking_behaviour[behavioural_features]
    .fillna(0)
    .astype(int)
)

print(
    "Remaining missing behavioural values:",
    ranking_behaviour[behavioural_features].isna().sum().sum()
)

Ranking events with no preceding behaviour: 43384
Remaining missing behavioural values: 0


In [ ]:
# Add session behavioural features to training, validation and test data

ranking_behaviour_features = ranking_behaviour[
    ["ranking_event_id"] + behavioural_features
]

train_data = train_data.merge(
    ranking_behaviour_features,
    on="ranking_event_id",
    how="left",
    validate="many_to_one"
)

val_data = val_data.merge(
    ranking_behaviour_features,
    on="ranking_event_id",
    how="left",
    validate="many_to_one"
)

test_data = test_data.merge(
    ranking_behaviour_features,
    on="ranking_event_id",
    how="left",
    validate="many_to_one"
)

print("Training rows:", len(train_data))
print("Validation rows:", len(val_data))
print("Test rows:", len(test_data))

print(
    "Missing behavioural values in training:",
    train_data[behavioural_features].isna().sum().sum()
)

print(
    "Missing behavioural values in validation:",
    val_data[behavioural_features].isna().sum().sum()
)

print(
    "Missing behavioural values in test:",
    test_data[behavioural_features].isna().sum().sum()
)

Training rows: 2539550
Validation rows: 543270
Test rows: 531310
Missing behavioural values in training: 0
Missing behavioural values in validation: 0
Missing behavioural values in test: 0


In [ ]:
# Save behavioural-feature checkpoints

train_data.to_parquet(
    f"{DATA_PATH}/train_preprocessed_behaviour.parquet",
    index=False
)

val_data.to_parquet(
    f"{DATA_PATH}/val_preprocessed_behaviour.parquet",
    index=False
)

test_data.to_parquet(
    f"{DATA_PATH}/test_preprocessed_behaviour.parquet",
    index=False
)

print("Behavioural-feature checkpoints saved.")

Behavioural-feature checkpoints saved.


In [ ]:
# Reconstruct ranking-event ID using the validated session_id + step definition

candidates["ranking_event_id"] = (
    candidates["session_id"].astype(str)
    + "_"
    + candidates["step"].astype(str)
)

print(
    "Unique candidate ranking events:",
    candidates["ranking_event_id"].nunique()
)

print(
    "Expected ranking events:",
    ranking_behaviour["ranking_event_id"].nunique()
)

candidate_ids = set(candidates["ranking_event_id"].dropna())
behaviour_ids = set(ranking_behaviour["ranking_event_id"].dropna())

print(
    "Matching ranking events:",
    len(candidate_ids & behaviour_ids)
)

print(
    "Candidate events without behavioural mapping:",
    len(candidate_ids - behaviour_ids)
)

print(
    "Behavioural events without candidates:",
    len(behaviour_ids - candidate_ids)
)

Unique candidate ranking events: 157726
Expected ranking events: 157809
Matching ranking events: 157726
Candidate events without behavioural mapping: 0
Behavioural events without candidates: 83


In [ ]:
# Price preprocessing pipeline

price_pipeline = Pipeline([
    ("log_transform", FunctionTransformer(np.log1p)),
    ("scale", StandardScaler())
])

In [ ]:
# Define the input features for LR

lr_features = [
    "candidate_price",
    "relative_price",
    "filter_match",
    "city_lr",
    "platform",
    "device"
] + behavioural_features

X_train_lr = train_data[lr_features]
X_val_lr = val_data[lr_features]
X_test_lr = test_data[lr_features]

y_train = train_data["clicked"]
y_val = val_data["clicked"]
y_test = test_data["clicked"]

print("LR features:", len(lr_features))
print("Training shape:", X_train_lr.shape)
print("Validation shape:", X_val_lr.shape)
print("Test shape:", X_test_lr.shape)

LR features: 15
Training shape: (2539550, 15)
Validation shape: (543270, 15)
Test shape: (531310, 15)


## Building the LR Preprocessing pipeline

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import (
    FunctionTransformer,
    StandardScaler,
    OneHotEncoder
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
# Define the model's sub-pipelines

price_pipeline = Pipeline([
    ("log_transform", FunctionTransformer(np.log1p)),
    ("scale", StandardScaler())
])

city_pipeline = Pipeline([
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    ))
])

In [ ]:
# Create the LR preprocessor

lr_preprocessor = ColumnTransformer(
    transformers=[
        (
            "price",
            price_pipeline,
            ["candidate_price"]
        ),
        (
            "relative_price",
            StandardScaler(),
            ["relative_price"]
        ),
        (
            "filter_match",
            "passthrough",
            ["filter_match"]
        ),
        (
            "city",
            city_pipeline,
            ["city_lr"]
        ),
        (
            "platform_device",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            ["platform", "device"]
        ),
        (
            "behaviour",
            StandardScaler(),
            behavioural_features
        )
    ]
)

In [ ]:
from sklearn.linear_model import SGDClassifier

# Complete Logistic Regression pipeline

lr_pipeline = Pipeline([
    ("preprocessor", lr_preprocessor),
    ("model", SGDClassifier(
        loss="log_loss",
        class_weight="balanced",
        max_iter=1000,
        tol=1e-3,
        random_state=RANDOM_STATE
    ))
])

lr_pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('price',
                                                  Pipeline(steps=[('log_transform',
                                                                   FunctionTransformer(func=<ufunc 'log1p'>)),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['candidate_price']),
                                                 ('relative_price',
                                                  StandardScaler(),
                                                  ['relative_price']),
                                                 ('filter_match', 'passthrough',
                                                  ['filter_match']),
                                                 ('city',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(h...
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['platform', 'device']),
                                                 ('behaviour', StandardScaler(),
                                                  ['count_sort_change',
                                                   'count_filter_selection',
                                                   'count_item_deals',
                                                   'count_item_image',
                                                   'count_item_info',
                                                   'count_item_rating',
                                                   'count_search_destination',
                                                   'count_search_item',
                                                   'count_search_poi'])])),
                ('model',
                 SGDClassifier(class_weight='balanced', loss='log_loss',
                               random_state=42))])

### Logistic Regression preprocessing completed

Logistic Regression uses candidate price, relative price, filter match, city, platform, device, and nine features representing preceding session behaviour. Candidate price is log-transformed to reduce skew, numerical features are standardised where appropriate, and categorical features are one-hot encoded. Cities occurring fewer than 50 times in the training data are grouped as Other to reduce dimensionality.

Preprocessing is defined within a scikit-learn pipeline and will be fitted on the training data only. The same fitted transformations will subsequently be applied to the validation and test data, preventing information leakage.

### LightGBM LambdaRank preprocessing

LightGBM LambdaRank uses the same 15 underlying predictors as Logistic Regression, ensuring a consistent basis for model comparison. However, numerical features do not require scaling and categorical features do not require one-hot encoding. `ranking_event_id` is retained to group candidates belonging to the same ranking event but is not used as a predictor.

Candidate position remains excluded to avoid incorporating Trivago's existing ranking and potential exposure bias.

In [ ]:
# Define LightGBM LambdaRank input features

lgbm_features = [
    "candidate_price",
    "relative_price",
    "filter_match",
    "city_lr",
    "platform",
    "device"
] + behavioural_features

X_train_lgbm = train_data[lgbm_features].copy()
X_val_lgbm = val_data[lgbm_features].copy()
X_test_lgbm = test_data[lgbm_features].copy()

y_train_lgbm = train_data[target].copy()
y_val_lgbm = val_data[target].copy()
y_test_lgbm = test_data[target].copy()

train_groups = train_data["ranking_event_id"].copy()
val_groups = val_data["ranking_event_id"].copy()
test_groups = test_data["ranking_event_id"].copy()

print("Features:", len(lgbm_features))
print("Training features:", X_train_lgbm.shape)
print("Validation features:", X_val_lgbm.shape)
print("Test features:", X_test_lgbm.shape)

print("\nRanking events:")
print("Training:", train_groups.nunique())
print("Validation:", val_groups.nunique())
print("Test:", test_groups.nunique())

Features: 15
Training features: (2539550, 15)
Validation features: (543270, 15)
Test features: (531310, 15)

Ranking events:
Training: 110716
Validation: 23737
Test: 23273


In [ ]:
# Prepare categorical features for LightGBM

lgbm_categorical_features = [
    "city_lr",
    "platform",
    "device"
]

for feature in lgbm_categorical_features:
    X_train_lgbm[feature] = X_train_lgbm[feature].astype("category")
    X_val_lgbm[feature] = pd.Categorical(
        X_val_lgbm[feature],
        categories=X_train_lgbm[feature].cat.categories
    )

    X_test_lgbm[feature] = pd.Categorical(
        X_test_lgbm[feature],
        categories=X_train_lgbm[feature].cat.categories
    )

print(X_train_lgbm.dtypes)

candidate_price                int64
relative_price               float64
filter_match                   int64
city_lr                     category
platform                    category
device                      category
count_sort_change              int64
count_filter_selection         int64
count_item_deals               int64
count_item_image               int64
count_item_info                int64
count_item_rating              int64
count_search_destination       int64
count_search_item              int64
count_search_poi               int64
dtype: object


In [ ]:
# Check categorical preparation

print(X_train_lgbm.dtypes)

for feature in lgbm_categorical_features:
    print(
        feature,
        "| categories:", len(X_train_lgbm[feature].cat.categories),
        "| unseen validation values:", X_val_lgbm[feature].isna().sum(),
        "| unseen test values:", X_test_lgbm[feature].isna().sum()
    )

candidate_price                int64
relative_price               float64
filter_match                   int64
city_lr                     category
platform                    category
device                      category
count_sort_change              int64
count_filter_selection         int64
count_item_deals               int64
count_item_image               int64
count_item_info                int64
count_item_rating              int64
count_search_destination       int64
count_search_item              int64
count_search_poi               int64
dtype: object
city_lr | categories: 5105 | unseen validation values: 0 | unseen test values: 0
platform | categories: 55 | unseen validation values: 0 | unseen test values: 0
device | categories: 3 | unseen validation values: 0 | unseen test values: 0


**In the next sequence the following is done:**
- Sort the rows so all candidates belonging to the same ranking_event_id are next to each other.
- Tell LambdaRank the group sizes, e.g. [25, 18, 31, ...].
- Remove ranking_event_id from the predictive features, because the ID itself carries no useful predictive information.

In [ ]:
# Keep candidates from each ranking event together

train_order = train_groups.sort_values().index
val_order = val_groups.sort_values().index
test_order = test_groups.sort_values().index

X_train_lgbm = X_train_lgbm.loc[train_order].reset_index(drop=True)
y_train_lgbm = y_train_lgbm.loc[train_order].reset_index(drop=True)
train_groups = train_groups.loc[train_order].reset_index(drop=True)

X_val_lgbm = X_val_lgbm.loc[val_order].reset_index(drop=True)
y_val_lgbm = y_val_lgbm.loc[val_order].reset_index(drop=True)
val_groups = val_groups.loc[val_order].reset_index(drop=True)

X_test_lgbm = X_test_lgbm.loc[test_order].reset_index(drop=True)
y_test_lgbm = y_test_lgbm.loc[test_order].reset_index(drop=True)
test_groups = test_groups.loc[test_order].reset_index(drop=True)

print("Training ranking events:", train_groups.nunique())
print("Validation ranking events:", val_groups.nunique())
print("Test ranking events:", test_groups.nunique())

Training ranking events: 110716
Validation ranking events: 23737
Test ranking events: 23273


In [ ]:
# Create ranking group sizes for LambdaRank

train_group_sizes = (
    train_groups
    .value_counts(sort=False)
    .to_numpy()
)

val_group_sizes = (
    val_groups
    .value_counts(sort=False)
    .to_numpy()
)

test_group_sizes = (
    test_groups
    .value_counts(sort=False)
    .to_numpy()
)

print("Training groups:", len(train_group_sizes))
print("Validation groups:", len(val_group_sizes))
print("Test groups:", len(test_group_sizes))

print("\nCandidates:")
print("Training:", train_group_sizes.sum())
print("Validation:", val_group_sizes.sum())
print("Test:", test_group_sizes.sum())

print("\nFirst 10 training group sizes:")
print(train_group_sizes[:10])

Training groups: 110716
Validation groups: 23737
Test groups: 23273

Candidates:
Training: 2539550
Validation: 543270
Test: 531310

First 10 training group sizes:
[25 25 25  9  9  9 25 25 25 25]


In [ ]:
# Validate LambdaRank groups and targets

def positive_counts_by_group(groups, targets):
    return (
        pd.DataFrame({
            "ranking_event_id": groups,
            "clicked": targets
        })
        .groupby("ranking_event_id")["clicked"]
        .sum()
    )

train_positive_counts = positive_counts_by_group(
    train_groups, y_train_lgbm
)

val_positive_counts = positive_counts_by_group(
    val_groups, y_val_lgbm
)

test_positive_counts = positive_counts_by_group(
    test_groups, y_test_lgbm
)

print(
    "Training groups with exactly one positive:",
    (train_positive_counts == 1).sum(),
    "/",
    len(train_positive_counts)
)

print(
    "Validation groups with exactly one positive:",
    (val_positive_counts == 1).sum(),
    "/",
    len(val_positive_counts)
)

print(
    "Test groups with exactly one positive:",
    (test_positive_counts == 1).sum(),
    "/",
    len(test_positive_counts)
)

print("\nGroup sizes sum correctly:")
print("Training:", train_group_sizes.sum() == len(X_train_lgbm))
print("Validation:", val_group_sizes.sum() == len(X_val_lgbm))
print("Test:", test_group_sizes.sum() == len(X_test_lgbm))

Training groups with exactly one positive: 110716 / 110716
Validation groups with exactly one positive: 23737 / 23737
Test groups with exactly one positive: 23273 / 23273

Group sizes sum correctly:
Training: True
Validation: True
Test: True


The LightGBM LambdaRank preprocessing is complete.

Next, we preprocess for the Dense Neural Network / MLP model.

Like Logistic Regression, the MLP requires scaled numerical values and numerical representations of categorical features. The Logistic Regression preprocessing can therefore be reused, ensuring that both models receive the same underlying predictors and a consistent feature representation.

## Wide & Deep Neural Network preprocessing

The Wide & Deep model also receives the same 15 underlying predictors used by the other models. However, these predictors are divided between the wide and deep paths according to their role.

The wide path contains `relative_price` and `filter_match`, which directly express candidate information in relation to the current ranking context. The deep path contains candidate price, contextual features and preceding session-behaviour features, allowing the neural network to learn more complex relationships between them.

The paths are kept non-overlapping so that the combined model still receives the same 15 underlying predictors as the other models rather than introducing additional predictive information.

In [ ]:
# Define Wide & Deep input features

wide_features = [
    "relative_price",
    "filter_match"
]

deep_features = [
    "candidate_price",
    "city_lr",
    "platform",
    "device"
] + behavioural_features

X_train_wide = train_data[wide_features].copy()
X_val_wide = val_data[wide_features].copy()
X_test_wide = test_data[wide_features].copy()

X_train_deep = train_data[deep_features].copy()
X_val_deep = val_data[deep_features].copy()
X_test_deep = test_data[deep_features].copy()

y_train_wd = train_data[target].copy()
y_val_wd = val_data[target].copy()
y_test_wd = test_data[target].copy()

print("Total predictors:", len(wide_features) + len(deep_features))

print("\nWide inputs:")
print("Training:", X_train_wide.shape)
print("Validation:", X_val_wide.shape)
print("Test:", X_test_wide.shape)

print("\nDeep inputs:")
print("Training:", X_train_deep.shape)
print("Validation:", X_val_deep.shape)
print("Test:", X_test_deep.shape)

Total predictors: 15

Wide inputs:
Training: (2539550, 2)
Validation: (543270, 2)
Test: (531310, 2)

Deep inputs:
Training: (2539550, 13)
Validation: (543270, 13)
Test: (531310, 13)


In [ ]:
# Wide path preprocessing

wide_preprocessor = ColumnTransformer(
    transformers=[
        (
            "relative_price",
            StandardScaler(),
            ["relative_price"]
        ),
        (
            "filter_match",
            "passthrough",
            ["filter_match"]
        )
    ]
)

In [ ]:
# Deep path preprocessing

deep_preprocessor = ColumnTransformer(
    transformers=[
        (
            "price",
            price_pipeline,
            ["candidate_price"]
        ),
        (
            "city",
            city_pipeline,
            ["city_lr"]
        ),
        (
            "platform_device",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            ["platform", "device"]
        ),
        (
            "behaviour",
            StandardScaler(),
            behavioural_features
        )
    ]
)

In [ ]:
# Save final preprocessed training dataset

train_data.to_parquet(
    f"{DATA_PATH}/train_preprocessed.parquet",
    index=False
)

print("Final training dataset saved.")
print("Rows:", len(train_data))
print("Sessions:", train_data["session_id"].nunique())
print("Ranking events:", train_data["ranking_event_id"].nunique())

Final training dataset saved.
Rows: 2539550
Sessions: 57849
Ranking events: 110716


In [ ]:
# Save final preprocessed validation dataset

val_data.to_parquet(
    f"{DATA_PATH}/val_preprocessed.parquet",
    index=False
)

print("Final validation dataset saved.")
print("Rows:", len(val_data))
print("Sessions:", val_data["session_id"].nunique())
print("Ranking events:", val_data["ranking_event_id"].nunique())

Final validation dataset saved.
Rows: 543270
Sessions: 12396
Ranking events: 23737


In [ ]:
# Save final preprocessed test dataset

test_data.to_parquet(
    f"{DATA_PATH}/test_preprocessed.parquet",
    index=False
)

print("Final test dataset saved.")
print("Rows:", len(test_data))
print("Sessions:", test_data["session_id"].nunique())
print("Ranking events:", test_data["ranking_event_id"].nunique())

Final test dataset saved.
Rows: 531310
Sessions: 12397
Ranking events: 23273
